# Auditoria visual de data augmentation

Este notebook verifica se A1 e A2 são **plausíveis e corretamente pareados** antes de qualquer treinamento. Ele não escolhe o melhor método e abre somente oito imagens fixas do treino. Validação, teste e FUSeg permanecem fechados.

## Critérios da auditoria

A auditoria inspeciona a plausibilidade das transformações, o alinhamento entre imagem e máscara e a retenção do primeiro plano.

## Braços pré-registrados

- **A0:** sem augmentation.
- **A1:** flips e affine moderado (rotação, escala e translação), sempre aplicado de forma idêntica à imagem e à máscara.
- **A2:** A1 mais, com probabilidade 0,5, exatamente uma alteração fotométrica leve.

Alterações fotométricas atingem somente a imagem; a máscara não muda.

## 1. Montar o Drive e localizar o projeto

In [ ]:
from pathlib import Path
import os, shutil, subprocess, sys

try:
    from google.colab import drive

    drive.mount("/content/drive")
except ImportError:
    print("Execução fora do Colab.")

candidates = [
    Path("/content/drive/MyDrive/TCC/Projeto_TCC_Lucas"),
    Path("/content/drive/MyDrive/Projeto_TCC_Lucas"),
]
PROJECT_ROOT = next((p for p in candidates if p.is_dir()), Path.cwd())
CONFIG_RELATIVE = Path("configs/augmentation_audit.yaml")
if not (PROJECT_ROOT / CONFIG_RELATIVE).is_file():
    raise FileNotFoundError("Configuração da auditoria de aumento de dados não encontrada.")
os.chdir(PROJECT_ROOT)
SOURCE_ROOT = PROJECT_ROOT / "src"
os.environ["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + os.environ.get("PYTHONPATH", "")
if str(SOURCE_ROOT) not in sys.path:
    sys.path.insert(0, str(SOURCE_ROOT))
print("Projeto:", PROJECT_ROOT)

## 2. Instalar o ambiente registrado

In [ ]:
result = subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-colab.txt"],
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 3. Copiar os dados para o disco temporário do Colab

In [ ]:
DATA_ROOT = Path("/content/tcc_data") if Path("/content").is_dir() else PROJECT_ROOT
if DATA_ROOT != PROJECT_ROOT:
    shutil.copytree(PROJECT_ROOT / "Dataset", DATA_ROOT / "Dataset", dirs_exist_ok=True)
print("Dados:", DATA_ROOT / "Dataset")

## 4. Conferir o protocolo antes de ver as imagens

Esta célula confirma que as faixas já estavam registradas e que a auditoria só pode ler treino.

In [ ]:
import yaml
from IPython.display import display

CONFIG = yaml.safe_load((PROJECT_ROOT / CONFIG_RELATIVE).read_text(encoding="utf-8"))
display(CONFIG)
assert CONFIG["runtime"]["allowed_split"] == "train"
assert CONFIG["runtime"]["model_training_allowed"] is False
assert CONFIG["pipelines"]["A1"]["affine"]["shear_degrees"] == [0.0, 0.0]

## 5. Testar as funções isoladamente

Os testes confirmam reprodutibilidade, máscara binária, mesma dimensão entre imagem e máscara, proibição de shear e no máximo uma transformação fotométrica em A2.

In [ ]:
test_env = os.environ.copy()
test_env["PYTHONPATH"] = str(SOURCE_ROOT) + os.pathsep + test_env.get("PYTHONPATH", "")
result = subprocess.run(
    [sys.executable, "-m", "pytest", "-q", "tests/test_augmentation.py"],
    cwd=PROJECT_ROOT,
    env=test_env,
    text=True,
    capture_output=True,
)
print(result.stdout)
print(result.stderr)
result.check_returncode()

## 6. Gerar os painéis e diagnósticos

Cada linha contém um caso fixo. Há uma imagem original, três sorteios de A1 e três de A2. O verde mostra onde a máscara ficou depois da transformação pareada.

In [ ]:
command = [
    sys.executable,
    "scripts/audit_augmentation.py",
    "--project-root",
    str(PROJECT_ROOT),
    "--data-root",
    str(DATA_ROOT),
    "--config",
    str(CONFIG_RELATIVE),
]
process = subprocess.Popen(
    command,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    cwd=PROJECT_ROOT,
    env=test_env,
)
for line in process.stdout:
    print(line, end="")
returncode = process.wait()
print("returncode=", returncode)
if returncode != 0:
    raise subprocess.CalledProcessError(returncode, command)

## 7. Visualizar e interpretar as saídas

In [ ]:
from PIL import Image
import pandas as pd

AUDIT_ROOT = PROJECT_ROOT / CONFIG["runtime"]["output_root"]
OUTPUT_DIR = max((p for p in AUDIT_ROOT.iterdir() if p.is_dir()), key=lambda p: p.stat().st_mtime)
print("Saídas:", OUTPUT_DIR)
display(Image.open(OUTPUT_DIR / "augmentation_panel.png"))
display(Image.open(OUTPUT_DIR / "augmentation_mask_panel.png"))
diagnostics = pd.read_csv(OUTPUT_DIR / "augmentation_diagnostics.csv")
display(diagnostics.groupby("pipeline").mean(numeric_only=True))
display(diagnostics[diagnostics["requires_review"] == True])

## O que observar

1. A máscara verde deve acompanhar exatamente a ferida após flips/affine.
2. Nenhuma máscara deve virar cinza: seus valores continuam binários.
3. A ferida não deve ser apagada ou cortada de modo severo.
4. A2 pode mudar cor/luz levemente, sem criar cores clinicamente absurdas.
5. Bordas pretas pequenas podem surgir pela rotação; valores extensos merecem revisão.

Uma variante visualmente bonita não é evidência de melhor segmentação.

## Artefatos da execução

Arquive `augmentation_audit_summary.json`, `augmentation_diagnostics.csv`, `augmentation_panel.png` e `augmentation_mask_panel.png`. A auditoria precede a comparação A0/A1/A2.